# Parser combinators in Python

This notebook translates the TypeScript parser-combinator example into **Python**.

To make the style more convincing, the parser combinator API uses **operator overloading** where it helps:

- `p1 | p2` means: try `p1`, otherwise `p2`
- `p1 + p2` means: parse `p1` followed by `p2`
- `p.map(f)` transforms a parser result

The final parser supports:

- variables consisting of letters
- numbers consisting of digits
- binary operators `+`, `-`, `*`, `/`, and `^`
- unary functions `exp(...)` and `log(...)`
- parentheses `(` and `)`

The output is an **abstract syntax tree (AST)**.

## 1. Core parser result types

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Generic, TypeVar, Union, List, Tuple, Any, Optional

T = TypeVar("T")
U = TypeVar("U")


@dataclass(frozen=True)
class ParseSuccess(Generic[T]):
    value: T
    rest: str


@dataclass(frozen=True)
class ParseFailure:
    error: str


ParseResult = Union[ParseSuccess[T], ParseFailure]

## 2. Parser class with combinators and operator overloading

The `Parser` class wraps a function `str -> ParseResult`.

We implement:

- `map`
- `then`
- `or_else`
- `|` as choice
- `+` as sequencing

In [2]:
class Parser(Generic[T]):
    def __init__(self, fn: Callable[[str], ParseResult[T]]):
        self.fn = fn

    def __call__(self, input: str) -> ParseResult[T]:
        return self.fn(input)

    def map(self, f: Callable[[T], U]) -> "Parser[U]":
        def parse(input: str) -> ParseResult[U]:
            result = self(input)
            if isinstance(result, ParseFailure):
                return result
            return ParseSuccess(f(result.value), result.rest)
        return Parser(parse)

    def then(self, other: "Parser[U]") -> "Parser[Tuple[T, U]]":
        def parse(input: str) -> ParseResult[Tuple[T, U]]:
            r1 = self(input)
            if isinstance(r1, ParseFailure):
                return r1

            r2 = other(r1.rest)
            if isinstance(r2, ParseFailure):
                return r2

            return ParseSuccess((r1.value, r2.value), r2.rest)
        return Parser(parse)

    def or_else(self, other: "Parser[T]") -> "Parser[T]":
        def parse(input: str) -> ParseResult[T]:
            r1 = self(input)
            if isinstance(r1, ParseSuccess):
                return r1
            return other(input)
        return Parser(parse)

    def __or__(self, other: "Parser[T]") -> "Parser[T]":
        return self.or_else(other)

    def __add__(self, other: "Parser[U]") -> "Parser[Tuple[T, U]]":
        return self.then(other)

## 3. Primitive parsers

In [3]:
def pure(value: T) -> Parser[T]:
    return Parser(lambda input: ParseSuccess(value, input))


def fail(error: str) -> Parser[Any]:
    return Parser(lambda input: ParseFailure(error))


def char(expected: str) -> Parser[str]:
    def parse(input: str) -> ParseResult[str]:
        if input.startswith(expected):
            return ParseSuccess(expected, input[len(expected):])
        return ParseFailure(f'Expected "{expected}"')
    return Parser(parse)


def string(expected: str) -> Parser[str]:
    def parse(input: str) -> ParseResult[str]:
        if input.startswith(expected):
            return ParseSuccess(expected, input[len(expected):])
        return ParseFailure(f'Expected "{expected}"')
    return Parser(parse)


def satisfy(predicate: Callable[[str], bool], error_message: str) -> Parser[str]:
    def parse(input: str) -> ParseResult[str]:
        if input and predicate(input[0]):
            return ParseSuccess(input[0], input[1:])
        return ParseFailure(error_message)
    return Parser(parse)

## 4. Standard combinators

In [4]:
def many(parser: Parser[T]) -> Parser[List[T]]:
    def parse(input: str) -> ParseResult[List[T]]:
        values: List[T] = []
        rest = input

        while True:
            result = parser(rest)
            if isinstance(result, ParseFailure):
                break

            if result.rest == rest:
                return ParseFailure("many() received a parser that consumed no input")

            values.append(result.value)
            rest = result.rest

        return ParseSuccess(values, rest)

    return Parser(parse)


def many1(parser: Parser[T]) -> Parser[List[T]]:
    def parse(input: str) -> ParseResult[List[T]]:
        first = parser(input)
        if isinstance(first, ParseFailure):
            return first

        rest = many(parser)(first.rest)
        if isinstance(rest, ParseFailure):
            return rest

        return ParseSuccess([first.value] + rest.value, rest.rest)

    return Parser(parse)


def lazy(thunk: Callable[[], Parser[T]]) -> Parser[T]:
    def parse(input: str) -> ParseResult[T]:
        return thunk()(input)
    return Parser(parse)


def between(left: Parser[Any], middle: Parser[T], right: Parser[Any]) -> Parser[T]:
    def parse(input: str) -> ParseResult[T]:
        l = left(input)
        if isinstance(l, ParseFailure):
            return l

        m = middle(l.rest)
        if isinstance(m, ParseFailure):
            return m

        r = right(m.rest)
        if isinstance(r, ParseFailure):
            return r

        return ParseSuccess(m.value, r.rest)

    return Parser(parse)

## 5. Whitespace helpers

In [5]:
letter = satisfy(str.isalpha, "Expected letter")
digit = satisfy(str.isdigit, "Expected digit")
whitespace = satisfy(str.isspace, "Expected whitespace")

spaces = many(whitespace)


def token(parser: Parser[T]) -> Parser[T]:
    def parse(input: str) -> ParseResult[T]:
        leading = spaces(input)
        if isinstance(leading, ParseFailure):
            return leading

        result = parser(leading.rest)
        if isinstance(result, ParseFailure):
            return result

        trailing = spaces(result.rest)
        if isinstance(trailing, ParseFailure):
            return trailing

        return ParseSuccess(result.value, trailing.rest)

    return Parser(parse)

## 6. AST types

In [6]:
@dataclass(frozen=True)
class Number:
    value: int


@dataclass(frozen=True)
class Variable:
    name: str


@dataclass(frozen=True)
class Binary:
    op: str
    left: "Expr"
    right: "Expr"


@dataclass(frozen=True)
class Call:
    fn: str
    arg: "Expr"


Expr = Union[Number, Variable, Binary, Call]

## 7. Lexical parsers

In [7]:
identifier: Parser[str] = many1(letter).map(lambda chars: "".join(chars))
number_literal: Parser[int] = many1(digit).map(lambda chars: int("".join(chars)))

number_expr: Parser[Expr] = token(number_literal).map(Number)
variable_expr: Parser[Expr] = token(identifier).map(Variable)

## 8. Operator helpers

We use:

1. function calls and parentheses
2. exponentiation `^` as right-associative
3. multiplication and division
4. addition and subtraction

In [8]:
def chain_left(
    value_parser: Parser[T],
    op_parser: Parser[Callable[[T, T], T]]
) -> Parser[T]:
    def parse(input: str) -> ParseResult[T]:
        first = value_parser(input)
        if isinstance(first, ParseFailure):
            return first

        acc = first.value
        rest = first.rest

        while True:
            op = op_parser(rest)
            if isinstance(op, ParseFailure):
                break

            nxt = value_parser(op.rest)
            if isinstance(nxt, ParseFailure):
                return ParseFailure("Expected expression after operator")

            acc = op.value(acc, nxt.value)
            rest = nxt.rest

        return ParseSuccess(acc, rest)

    return Parser(parse)


def chain_right(
    value_parser: Parser[T],
    op_parser: Parser[Callable[[T, T], T]]
) -> Parser[T]:
    def parse_from(input: str) -> ParseResult[T]:
        left = value_parser(input)
        if isinstance(left, ParseFailure):
            return left

        op = op_parser(left.rest)
        if isinstance(op, ParseFailure):
            return left

        right = parse_from(op.rest)
        if isinstance(right, ParseFailure):
            return right

        return ParseSuccess(op.value(left.value, right.value), right.rest)

    return Parser(parse_from)

## 9. AST-building operator parsers

In [9]:
add_op: Parser[Callable[[Expr, Expr], Expr]] = token(string("+")).map(
    lambda _: lambda a, b: Binary("+", a, b)
)

sub_op: Parser[Callable[[Expr, Expr], Expr]] = token(string("-")).map(
    lambda _: lambda a, b: Binary("-", a, b)
)

mul_op: Parser[Callable[[Expr, Expr], Expr]] = token(string("*")).map(
    lambda _: lambda a, b: Binary("*", a, b)
)

div_op: Parser[Callable[[Expr, Expr], Expr]] = token(string("/")).map(
    lambda _: lambda a, b: Binary("/", a, b)
)

pow_op: Parser[Callable[[Expr, Expr], Expr]] = token(string("^")).map(
    lambda _: lambda a, b: Binary("^", a, b)
)

## 10. Grammar

We implement the grammar

```text
expr    := term (("+" | "-") term)*
term    := power (("*" | "/") power)*
power   := primary ("^" power)?
primary := number
         | variable
         | "exp" "(" expr ")"
         | "log" "(" expr ")"
         | "(" expr ")"
```

In [10]:
expr: Optional[Parser[Expr]] = None

lparen = token(string("("))
rparen = token(string(")"))


def function_call(fn_name: str) -> Parser[Expr]:
    def parse(input: str) -> ParseResult[Expr]:
        fn = token(string(fn_name))(input)
        if isinstance(fn, ParseFailure):
            return fn

        assert expr is not None
        arg = between(lparen, lazy(lambda: expr), rparen)(fn.rest)
        if isinstance(arg, ParseFailure):
            return arg

        return ParseSuccess(Call(fn_name, arg.value), arg.rest)

    return Parser(parse)


def primary_parser(input: str) -> ParseResult[Expr]:
    global expr

    parsers: List[Parser[Expr]] = [
        function_call("exp"),
        function_call("log"),
        between(lparen, lazy(lambda: expr), rparen),
        number_expr,
        variable_expr,
    ]

    last_error = "Expected number, variable, function call, or parenthesized expression"

    for p in parsers:
        result = p(input)
        if isinstance(result, ParseSuccess):
            return result
        last_error = result.error

    return ParseFailure(last_error)


primary = Parser(primary_parser)
power = chain_right(primary, pow_op)
term = chain_left(power, mul_op | div_op)
expr = chain_left(term, add_op | sub_op)

## 11. Entry point

In [11]:
def parse_expression(input: str) -> ParseResult[Expr]:
    assert expr is not None

    result = expr(input)
    if isinstance(result, ParseFailure):
        return result

    trailing = spaces(result.rest)
    if isinstance(trailing, ParseFailure):
        return trailing

    if trailing.rest:
        return ParseFailure(f'Unexpected trailing input: "{trailing.rest}"')

    return ParseSuccess(result.value, "")

## 12. Example inputs

In [12]:
examples = [
    "42",
    "x",
    "a+b*c",
    "(a+b)*c",
    "x^y^z",
    "exp(x)",
    "log(123)",
    "exp(x + 2) / log(y)",
    "a + b * exp(c ^ d) - log(z)",
]

for input_text in examples:
    print("\nINPUT:", input_text)
    print(parse_expression(input_text))


INPUT: 42
ParseSuccess(value=Number(value=42), rest='')

INPUT: x
ParseSuccess(value=Variable(name='x'), rest='')

INPUT: a+b*c
ParseSuccess(value=Binary(op='+', left=Variable(name='a'), right=Binary(op='*', left=Variable(name='b'), right=Variable(name='c'))), rest='')

INPUT: (a+b)*c
ParseSuccess(value=Binary(op='*', left=Binary(op='+', left=Variable(name='a'), right=Variable(name='b')), right=Variable(name='c')), rest='')

INPUT: x^y^z
ParseSuccess(value=Binary(op='^', left=Variable(name='x'), right=Binary(op='^', left=Variable(name='y'), right=Variable(name='z'))), rest='')

INPUT: exp(x)
ParseSuccess(value=Call(fn='exp', arg=Variable(name='x')), rest='')

INPUT: log(123)
ParseSuccess(value=Call(fn='log', arg=Number(value=123)), rest='')

INPUT: exp(x + 2) / log(y)
ParseSuccess(value=Binary(op='/', left=Call(fn='exp', arg=Binary(op='+', left=Variable(name='x'), right=Number(value=2))), right=Call(fn='log', arg=Variable(name='y'))), rest='')

INPUT: a + b * exp(c ^ d) - log(z)
Parse

## 13. Optional pretty printer for ASTs

In [13]:
def ast_to_dict(expr: Expr) -> dict:
    if isinstance(expr, Number):
        return {"type": "Number", "value": expr.value}
    if isinstance(expr, Variable):
        return {"type": "Variable", "name": expr.name}
    if isinstance(expr, Binary):
        return {
            "type": "Binary",
            "op": expr.op,
            "left": ast_to_dict(expr.left),
            "right": ast_to_dict(expr.right),
        }
    if isinstance(expr, Call):
        return {
            "type": "Call",
            "fn": expr.fn,
            "arg": ast_to_dict(expr.arg),
        }
    raise TypeError(f"Unknown Expr: {expr!r}")


for input_text in examples:
    result = parse_expression(input_text)
    print("\nINPUT:", input_text)
    if isinstance(result, ParseSuccess):
        import json
        print(json.dumps(ast_to_dict(result.value), indent=2))
    else:
        print(result.error)


INPUT: 42
{
  "type": "Number",
  "value": 42
}

INPUT: x
{
  "type": "Variable",
  "name": "x"
}

INPUT: a+b*c
{
  "type": "Binary",
  "op": "+",
  "left": {
    "type": "Variable",
    "name": "a"
  },
  "right": {
    "type": "Binary",
    "op": "*",
    "left": {
      "type": "Variable",
      "name": "b"
    },
    "right": {
      "type": "Variable",
      "name": "c"
    }
  }
}

INPUT: (a+b)*c
{
  "type": "Binary",
  "op": "*",
  "left": {
    "type": "Binary",
    "op": "+",
    "left": {
      "type": "Variable",
      "name": "a"
    },
    "right": {
      "type": "Variable",
      "name": "b"
    }
  },
  "right": {
    "type": "Variable",
    "name": "c"
  }
}

INPUT: x^y^z
{
  "type": "Binary",
  "op": "^",
  "left": {
    "type": "Variable",
    "name": "x"
  },
  "right": {
    "type": "Binary",
    "op": "^",
    "left": {
      "type": "Variable",
      "name": "y"
    },
    "right": {
      "type": "Variable",
      "name": "z"
    }
  }
}

INPUT: exp(x)
{
  "typ

## 14. Why Python feels nice here

This version is a good fit for Python because parser combinators can use:

- first-class functions
- lightweight classes
- dataclasses for AST nodes
- operator overloading for elegant composition

For example:

- `mul_op | div_op` expresses choice
- `many1(letter).map(...)` expresses transformation cleanly

If you want, the next step would be to overload even more operators and create a more DSL-like notation.